# 2 · Test all controllers (Colab)

Run this after training is finished, when all runs are in the Drive results folder.
If some runs were trained on your PC, copy them into the same folder first.

It does three things, all **resumable** (if Colab disconnects, run all cells again):
1. tunes the actuated baseline on the **validation** files (once per scenario);
2. tests fixed, actuated, random and every trained model (FP32 and INT8) on the 10 **test** files;
3. makes the results table and charts, and measures policy size and speed.

Runtime type: **CPU**. A GPU is not used.

## 1. Settings

In [ ]:
SCENARIOS = ["cologne1", "ingolstadt1", "cologne3"]
INT8      = True              # also test the 8-bit (quantised) copy of each policy
WORKERS   = None              # None = one simulation per CPU core

RESULTS = "/content/drive/MyDrive/icarc_results"
REPO    = "https://github.com/Konara98/icarc_dataset_scripts.git"

## 2. Setup

In [ ]:
# Setup: Google Drive, code, packages. Takes about 1-2 minutes.
import os, subprocess, sys, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(RESULTS, exist_ok=True)
os.environ["TSC_RESULTS"] = RESULTS          # every script saves into this Drive folder

CODE = "/content/icarc_dataset_scripts"
if os.path.exists(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
os.chdir(CODE)

# Same versions as the local PC, so results from both machines are comparable.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eclipse-sumo==1.27.1", "libsumo==1.27.1",
                "sumo-rl==1.4.5", "stable-baselines3==2.9.0"], check=True)

CPUS = os.cpu_count()
print(f"code: {CODE} ({subprocess.run(['git', 'log', '-1', '--format=%h %cd'], capture_output=True, text=True).stdout.strip()})")
print(f"results: {RESULTS}")
print(f"CPU cores: {CPUS}")

In [ ]:
# Which trained runs are in the results folder? (only these get tested)
import glob, json
from collections import Counter
runs = Counter()
for cfg in glob.glob(os.path.join(RESULTS, "*", "*", "seed*", "config.json")):
    s, tag = cfg.split(os.sep)[-4], cfg.split(os.sep)[-3]
    c = json.load(open(cfg))
    ok = "finished" in c and os.path.exists(os.path.join(os.path.dirname(cfg), "best_model.zip"))
    runs[(s, tag, "finished" if ok else "NOT finished")] += 1
for (s, tag, state), n in sorted(runs.items()):
    print(f"{s:12} {tag:26} {n} seeds {state}")
print("\nUnfinished runs still have a best_model.zip and WILL be tested; finish training first if any are listed.")

## 3. Tune the actuated baseline (validation files only)

This is skipped for scenarios that are already tuned. It takes about 5 minutes per scenario.

In [ ]:
workers = WORKERS or CPUS
todo = [s for s in SCENARIOS if not os.path.exists(os.path.join(RESULTS, s, "actuated_params.json"))]
if todo:
    r = subprocess.run([sys.executable, "evaluate.py", "--tune-actuated", "--scenarios", *todo,
                        "--workers", str(workers)], capture_output=True, text=True)
    print(r.stdout[-6000:] or r.stderr[-3000:])
else:
    print("already tuned:", SCENARIOS)

## 4. Test every controller on every test file

One row is added to `test_episodes.csv` per (controller, seed, file); finished rows are skipped on a re-run.
The last lines of the output are the summary tables.

In [ ]:
workers = WORKERS or CPUS
cmd = [sys.executable, "evaluate.py", "--scenarios", *SCENARIOS, "--workers", str(workers)] + (["--int8"] if INT8 else [])
log_path = os.path.join(RESULTS, "evaluate.log")
with open(log_path, "w") as fh:
    code_ = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT).returncode
lines = open(log_path).read().splitlines()
summary_start = next((i for i, l in enumerate(lines) if l.startswith("== ")), max(0, len(lines) - 40))
print("\n".join(lines[:1] + lines[summary_start:]) if code_ == 0 else "\n".join(lines[-40:]))

## 5. Results table: average total waiting per vehicle (s)

Mean ± std over the 5 training seeds. `x0.7` and `x1.3` are traffic levels never seen in
training, and `real` is the original hour. `vs fixed` is the % less waiting than fixed-time
control (positive = better).

In [ ]:
import pandas as pd
summ = pd.read_csv(os.path.join(RESULTS, "test_summary.csv"))
summ["precision"] = summ["precision"].fillna("")
summ["cell"] = [f"{m:.1f} ± {s:.1f}" if pd.notna(s) else f"{m:.1f}"
                for m, s in zip(summ.avg_total_wait_mean, summ.avg_total_wait_std)]
order = ["x0.7", "x1.0", "x1.3", "real", "all"]
for s in SCENARIOS:
    t = summ[summ.scenario == s]
    if t.empty:
        continue
    wait = t.pivot_table(index=["controller", "precision"], columns="demand", values="cell", aggfunc="first")
    gain = t.pivot_table(index=["controller", "precision"], columns="demand", values="vs_fixed_pct", aggfunc="first")
    fin = t.pivot_table(index=["controller", "precision"], columns="demand", values="finished_pct_mean", aggfunc="first")
    print(f"\n==== {s}: total wait per vehicle, s (mean ± std over seeds)")
    display(wait[[c for c in order if c in wait.columns]])
    print(f"{s}: % less waiting than fixed-time")
    display(gain[[c for c in order if c in gain.columns]].round(1))
    print(f"{s}: % of vehicles that finished their trip")
    display(fin[[c for c in order if c in fin.columns]].round(1))

## 6. Chart: does each controller keep its advantage on unseen traffic?

Total wait per vehicle at each demand level (FP32 policies, mean over seeds; the whiskers show
the spread between seeds). The random policy is left out because its bars would dwarf the others.
It is in the table above. Lower is better.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

COLORS = {"dqn": "#2a78d6", "ddqn": "#eb6834", "ppo": "#1baf7a", "a2c": "#eda100",
          "actuated": "#e87ba4", "fixed": "#6250d6"}
NAMES = {"dqn": "DQN", "ddqn": "Double DQN", "ppo": "PPO", "a2c": "A2C", "actuated": "actuated", "fixed": "fixed-time"}
levels = ["x0.7", "x1.0", "x1.3", "real"]
main = [c for c in summ.controller.unique() if c.endswith("_64x64_variants")]
series = ["fixed", "actuated"] + sorted(main, key=lambda c: list(COLORS).index(c.split("_")[0]))

fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(5.6 * len(SCENARIOS), 4), squeeze=False)
for ax, s in zip(axes[0], SCENARIOS):
    t = summ[(summ.scenario == s) & (summ.precision.isin(["", "fp32"]))]
    width = 0.8 / len(series)
    for k, c in enumerate(series):
        row = t[t.controller == c].set_index("demand").reindex(levels)
        key = c.split("_")[0]
        x = np.arange(len(levels)) + (k - (len(series) - 1) / 2) * width
        ax.bar(x, row.avg_total_wait_mean, width * 0.9, color=COLORS[key], label=NAMES[key],
               yerr=row.avg_total_wait_std.fillna(0), error_kw=dict(lw=1, ecolor="#52514e", capsize=0))
    ax.set_xticks(range(len(levels)), ["0.7× (unseen)", "1.0×", "1.3× (unseen)", "real hour"])
    ax.set_title(s, loc="left", fontsize=11)
    ax.set_ylabel("total wait per vehicle (s)")
    ax.grid(axis="y", color="#e6e5e0", lw=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(series), frameon=False, fontsize=9)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

## 7. Does 8-bit compression change the result?

INT8 minus FP32 total wait, per algorithm and demand level, in seconds (positive = INT8 is worse).
Look especially at 1.3×, where small decision changes matter most.

In [ ]:
if INT8:
    for s in SCENARIOS:
        t = summ[(summ.scenario == s) & summ.precision.isin(["fp32", "int8"])]
        if t.empty:
            continue
        p = t.pivot_table(index=["controller", "demand"], columns="precision", values="avg_total_wait_mean")
        d = (p["int8"] - p["fp32"]).unstack("demand")
        print(f"\n==== {s}: INT8 - FP32 total wait (s)")
        display(d[[c for c in order if c in d.columns]].round(2))

## 8. Policy size and speed

Colab's CPU is **not** roadside hardware. Use this table only to compare configurations with each
other. For the paper, run `python measure_policy.py` on the actual device as well.

In [ ]:
r = subprocess.run([sys.executable, "measure_policy.py"], capture_output=True, text=True)
print(r.stdout or r.stderr[-3000:])

## 9. Download everything as one zip (optional, models included)

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/icarc_results", "zip", RESULTS)
print(f"{os.path.getsize(zip_path) / 1e6:.1f} MB")
files.download(zip_path)